# Mid QFDet Stream 2
LLVIP(RGB) + LLVIP(Thermal)

In [ ]:
import subprocess, sys
for pkg in ['ultralytics', 'pandas', 'matplotlib', 'ensemble_boxes', 'optuna']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Packages OK')

In [ ]:
import os, sys, gc, math, time, random, json
sys.path.insert(0, '/root/AIP491')
from train_common import *

print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === Kiem tra cau truc thu muc data ===
import os
def show_tree(path, depth=3, _d=0):
    if _d > depth or not os.path.exists(path):
        return
    entries = sorted(os.listdir(path))
    files = [e for e in entries if os.path.isfile(os.path.join(path, e))]
    dirs  = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    pad = '  ' * _d
    for d in dirs[:8]:
        print(f'{pad}[{d}/]')
        show_tree(os.path.join(path, d), depth, _d+1)
    if files:
        print(f'{pad}  ({len(files)} files, e.g. {files[0]})')

print(f'RGBT_DATA_DIR = {RGBT_DATA_DIR}')
show_tree(RGBT_DATA_DIR, depth=3)

In [ ]:
STREAM = 2
RGB_BACKBONE_PATH = STREAM_CONFIGS[STREAM]['rgb']
THERMAL_BACKBONE_PATH = STREAM_CONFIGS[STREAM]['thr']
OUTPUT_DIR = os.path.join(BASE_DIR, 'Mid-stage', 'outputs', 'qfdet_stream2')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEEDS = [42, 777, 123]
NUM_EPOCHS = 50
BATCH_SIZE = 8
LR = 5e-4
WEIGHT_DECAY = 1e-3
WARMUP_EPOCHS = 5
PATIENCE = 2

print(f'Stream {STREAM}: {STREAM_CONFIGS[STREAM]["desc"]}')

In [ ]:
class PoolUpsample(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels), nn.SiLU()
        )

    def forward(self, feat):
        pooled = self.pool(feat)
        upsampled = F.interpolate(pooled, size=feat.shape[2:], mode='bilinear', align_corners=False)
        return self.reduce(torch.cat([feat, upsampled], dim=1))


class QualityAwareFusion(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        mid = max(in_channels // 4, 16)
        self.q_rgb = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.SiLU(),
            nn.Conv2d(mid, 1, 1), nn.Sigmoid()
        )
        self.q_thr = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.SiLU(),
            nn.Conv2d(mid, 1, 1), nn.Sigmoid()
        )
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels), nn.SiLU()
        )

    def forward(self, rgb_feat, thr_feat):
        q_rgb = self.q_rgb(rgb_feat)  # [B, 1, H, W]
        q_thr = self.q_thr(thr_feat)  # [B, 1, H, W]
        w = torch.softmax(torch.cat([q_rgb, q_thr], dim=1), dim=1)  # [B, 2, H, W]
        w_rgb = w[:, 0:1, :, :]
        w_thr = w[:, 1:2, :, :]
        fused = (1 + w_rgb) * rgb_feat + (1 + w_thr) * thr_feat
        return self.proj(fused)


class RGBTFusionDetector(nn.Module):
    EXTRACT_LAYERS = {4: 64, 6: 128, 9: 256}

    def __init__(self, rgb_backbone, thermal_backbone, nc=1, freeze_backbones=True):
        super().__init__()
        self.rgb_stream = rgb_backbone
        self.thermal_stream = thermal_backbone

        if freeze_backbones:
            for p in self.rgb_stream.parameters():
                p.requires_grad = False
            for p in self.thermal_stream.parameters():
                p.requires_grad = False

        self.pool_rgb_p3 = PoolUpsample(64)
        self.pool_rgb_p4 = PoolUpsample(128)
        self.pool_rgb_p5 = PoolUpsample(256)
        self.pool_thr_p3 = PoolUpsample(64)
        self.pool_thr_p4 = PoolUpsample(128)
        self.pool_thr_p5 = PoolUpsample(256)

        self.fuse_p3 = QualityAwareFusion(64)
        self.fuse_p4 = QualityAwareFusion(128)
        self.fuse_p5 = QualityAwareFusion(256)

        self.upsample   = nn.Upsample(scale_factor=2, mode='nearest')
        self.td_c2f_p4  = C2f(256 + 128, 128, n=1, shortcut=False)
        self.td_c2f_p3  = C2f(128 + 64, 64, n=1, shortcut=False)
        self.bu_conv_p4 = Conv(64, 64, 3, 2)
        self.bu_c2f_p4  = C2f(64 + 128, 128, n=1, shortcut=False)
        self.bu_conv_p5 = Conv(128, 128, 3, 2)
        self.bu_c2f_p5  = C2f(128 + 256, 256, n=1, shortcut=False)

        self.detect = Detect(nc=nc, ch=(64, 128, 256))
        self.detect.stride = torch.tensor([8., 16., 32.])

    def _extract_features(self, stream, x):
        feats = {}
        for i, layer in enumerate(stream):
            x = layer(x)
            if i in self.EXTRACT_LAYERS:
                feats[i] = x
        return feats

    def forward(self, rgb, thermal):
        if self.training:
            r = random.random()
            if r < 0.1:
                rgb = torch.zeros_like(rgb)
            elif r < 0.2:
                thermal = torch.zeros_like(thermal)

        rgb_f = self._extract_features(self.rgb_stream, rgb)
        thr_f = self._extract_features(self.thermal_stream, thermal)

        rgb_p3 = self.pool_rgb_p3(rgb_f[4])
        rgb_p4 = self.pool_rgb_p4(rgb_f[6])
        rgb_p5 = self.pool_rgb_p5(rgb_f[9])
        thr_p3 = self.pool_thr_p3(thr_f[4])
        thr_p4 = self.pool_thr_p4(thr_f[6])
        thr_p5 = self.pool_thr_p5(thr_f[9])

        p3 = self.fuse_p3(rgb_p3, thr_p3)
        p4 = self.fuse_p4(rgb_p4, thr_p4)
        p5 = self.fuse_p5(rgb_p5, thr_p5)

        p4_td  = self.td_c2f_p4(torch.cat([self.upsample(p5), p4], dim=1))
        p3_out = self.td_c2f_p3(torch.cat([self.upsample(p4_td), p3], dim=1))

        p4_out = self.bu_c2f_p4(torch.cat([self.bu_conv_p4(p3_out), p4_td], dim=1))
        p5_out = self.bu_c2f_p5(torch.cat([self.bu_conv_p5(p4_out), p5], dim=1))

        return self.detect([p3_out, p4_out, p5_out])

print('RGBTFusionDetector (QFDet) OK')

In [ ]:
# === Optuna Bayesian Tuning (15 trials) ===
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

_tune_out = os.path.join(OUTPUT_DIR, 'tune_best_params.json')

if os.path.exists(_tune_out):
    print(f'Tune results found: {_tune_out}')
    with open(_tune_out) as _f:
        _best = json.load(_f)
    LR           = _best.get('lr', LR)
    WEIGHT_DECAY = _best.get('weight_decay', WEIGHT_DECAY)
    BATCH_SIZE   = _best.get('batch_size', BATCH_SIZE)
    WARMUP_EPOCHS = _best.get('warmup_epochs', WARMUP_EPOCHS)
    print('Loaded params:', _best)
else:
    FUSION_DATA_DIR = os.path.join(BASE_DIR, 'data', 'mid_yolo')
    _fyd = setup_mid_dataset(FUSION_DATA_DIR)

    def _tune_mid(trial):
        _lr   = trial.suggest_float('lr', 1e-5, 5e-3, log=True)
        _wd   = trial.suggest_float('weight_decay', 1e-4, 5e-3, log=True)
        _bs   = trial.suggest_categorical('batch_size', [16, 32])
        _wu   = trial.suggest_int('warmup_epochs', 2, 7)

        try:
            set_seed(42)
            _train_ds = RGBTDataset(_fyd, 'train', IMG_SIZE, blur_prob=0.0, flip_prob=0.5)
            _val_ds   = RGBTDataset(_fyd, 'val', IMG_SIZE)
            _train_ld = DataLoader(_train_ds, batch_size=_bs, shuffle=True,
                                   num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
            _val_ld   = DataLoader(_val_ds, batch_size=_bs, shuffle=False,
                                   num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)

            _rgb_bb = load_yolov8n_backbone(RGB_BACKBONE_PATH)
            _thr_bb = load_yolov8n_backbone(THERMAL_BACKBONE_PATH)
            _model  = RGBTFusionDetector(_rgb_bb, _thr_bb, nc=1, freeze_backbones=True).to(device)

            _optimizer = AdamW([p for p in _model.parameters() if p.requires_grad], lr=_lr, weight_decay=_wd)
            _criterion = v8DetectionLoss(LossModel(_model))

            _best_val = float('inf')
            for _ep in range(1, 10 + 1):
                _ = run_epoch(_model, _train_ld, _optimizer, _criterion, train=True)
                _vl = run_epoch(_model, _val_ld, _optimizer, _criterion, train=False)
                if _vl < _best_val:
                    _best_val = _vl

                trial.report(_vl, _ep)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            del _model, _rgb_bb, _thr_bb, _optimizer, _criterion
            torch.cuda.empty_cache(); gc.collect()
            return _best_val

        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print(f'  OOM at bs={_bs}, skip trial')
                torch.cuda.empty_cache(); gc.collect()
                return float('inf')
            raise

    _study = optuna.create_study(
        direction='minimize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
        study_name='qfdet_tune'
    )
    _study.optimize(_tune_mid, n_trials=15, show_progress_bar=True)

    _best = dict(_study.best_params)
    _best['best_val_loss'] = _study.best_value
    with open(_tune_out, 'w') as _f:
        json.dump(_best, _f, indent=2)

    LR           = _best.get('lr', LR)
    WEIGHT_DECAY = _best.get('weight_decay', WEIGHT_DECAY)
    BATCH_SIZE   = _best.get('batch_size', BATCH_SIZE)
    WARMUP_EPOCHS = _best.get('warmup_epochs', WARMUP_EPOCHS)
    print('BEST PARAMS:')
    for _k, _v in _best.items():
        print(f'  {_k}: {_v}')

In [ ]:
# === Training ===
# Augmentation: Random Gaussian Blur (chi RGB, p=0.5) + Random HFlip (ca 2, p=0.5)
FUSION_DATA_DIR = os.path.join(BASE_DIR, 'data', 'mid_yolo')
fusion_yolo_dir = setup_mid_dataset(FUSION_DATA_DIR)

train_ds = RGBTDataset(fusion_yolo_dir, 'train', IMG_SIZE, blur_prob=0.0, flip_prob=0.5)
val_ds   = RGBTDataset(fusion_yolo_dir, 'val', IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)

all_histories = {}
all_results = {}

for seed_idx, SEED in enumerate(SEEDS):
    seed_dir = os.path.join(OUTPUT_DIR, f'seed_{SEED}')
    os.makedirs(seed_dir, exist_ok=True)
    best_path = os.path.join(seed_dir, 'fusion_best.pt')

    if os.path.exists(best_path):
        ckpt = torch.load(best_path, map_location=device, weights_only=False)
        if 'metrics' in ckpt:
            all_results[SEED] = ckpt['metrics']
        all_histories[SEED] = ckpt.get('history', {})
        print(f'Seed {SEED}: checkpoint exists, skip.')
        continue

    print(f'\n{"="*60}')
    print(f'  SEED {SEED} ({seed_idx+1}/{len(SEEDS)})')
    print(f'{"="*60}')

    set_seed(SEED)
    rgb_bb = load_yolov8n_backbone(RGB_BACKBONE_PATH)
    thr_bb = load_yolov8n_backbone(THERMAL_BACKBONE_PATH)
    fusion_model = RGBTFusionDetector(rgb_bb, thr_bb, nc=1, freeze_backbones=True).to(device)

    trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in fusion_model.parameters())
    print(f'Params: {total:,} total | {trainable:,} trainable ({100*trainable/total:.1f}%)')

    optimizer = AdamW([p for p in fusion_model.parameters() if p.requires_grad],
                     lr=LR, weight_decay=WEIGHT_DECAY)

    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return 0.01 + (1.0 - 0.01) * epoch / max(WARMUP_EPOCHS, 1)
        progress = (epoch - WARMUP_EPOCHS) / max(NUM_EPOCHS - WARMUP_EPOCHS, 1)
        return 0.01 + 0.5 * (1.0 - 0.01) * (1 + math.cos(math.pi * progress))

    scheduler = LambdaLR(optimizer, lr_lambda)
    criterion = v8DetectionLoss(LossModel(fusion_model))

    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    best_val = float('inf')
    no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss = run_epoch(fusion_model, train_loader, optimizer, criterion, train=True)
        va_loss = run_epoch(fusion_model, val_loader, optimizer, criterion, train=False)
        scheduler.step()
        lr = scheduler.get_last_lr()[0]

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['lr'].append(lr)

        flag = ''
        if va_loss < best_val:
            best_val = va_loss
            no_improve = 0
            torch.save({
                'epoch': epoch, 'seed': SEED,
                'model_state_dict': fusion_model.state_dict(),
                'val_loss': va_loss, 'train_loss': tr_loss,
                'history': history
            }, best_path)
            flag = '  <- best'
        else:
            no_improve += 1
            flag = f'  (no improve {no_improve}/{PATIENCE})'

        elapsed = time.time() - t0
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  train={tr_loss:.4f}  val={va_loss:.4f}  '
              f'lr={lr:.2e}  {elapsed:.0f}s{flag}')

        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

    all_histories[SEED] = history
    del fusion_model, optimizer, scheduler, criterion, rgb_bb, thr_bb
    torch.cuda.empty_cache(); gc.collect()

print(f'Training done for {len(SEEDS)} seeds.')

In [ ]:
# === Evaluation ===
for SEED in SEEDS:
    seed_dir = os.path.join(OUTPUT_DIR, f'seed_{SEED}')
    best_path = os.path.join(seed_dir, 'fusion_best.pt')
    if SEED in all_results:
        continue
    if not os.path.exists(best_path):
        continue

    print(f'\nEvaluating seed {SEED}...')
    set_seed(SEED)
    rgb_bb = load_yolov8n_backbone(RGB_BACKBONE_PATH)
    thr_bb = load_yolov8n_backbone(THERMAL_BACKBONE_PATH)
    fusion_model = RGBTFusionDetector(rgb_bb, thr_bb, nc=1, freeze_backbones=True).to(device)

    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    fusion_model.load_state_dict(ckpt['model_state_dict'])

    metrics = evaluate_model(fusion_model, val_loader)
    all_results[SEED] = metrics
    ckpt['metrics'] = metrics
    torch.save(ckpt, best_path)

    print(f'  mAP@0.5: {metrics["map50"]:.4f} | mAP@.5:.95: {metrics["map50_95"]:.4f}')
    del fusion_model, rgb_bb, thr_bb
    torch.cuda.empty_cache(); gc.collect()

print_summary(all_results, SEEDS, title=f'Mid QFDet Stream {STREAM} -- {STREAM_CONFIGS[STREAM]["desc"]}')
plot_loss_curves(all_histories, SEEDS,
                 os.path.join(OUTPUT_DIR, 'loss_curves.png'),
                 title=f'Mid QFDet Stream {STREAM}')